# DETR — DEtection TRansformer

## 핵심 논문
- **DETR** (ECCV 2020, Carion et al.): Transformer 기반 end-to-end 객체 검출
- **Deformable DETR** (ICLR 2021): 멀티스케일 + deformable attention → 빠른 수렴
- **DAB-DETR / DN-DETR / DINO**: 이후 발전 계보

## YOLO 계열 vs DETR
| | YOLO 계열 (anchor-based) | DETR |
|---|---|---|
| Anchor | 사전 정의 anchor 필요 | anchor-free |
| NMS | 필요 (후처리) | 불필요 (set prediction) |
| 매칭 | Threshold 기반 | Hungarian 최적 매칭 |
| 구조 | CNN | CNN + Transformer Encoder-Decoder |
| 단점 | 하이퍼파라미터 많음 | 학습 느림 (500ep), 소형 객체 약함 |

## DETR 핵심 아이디어
1. **Object Query**: N개의 learnable 벡터 → 각각 하나의 객체를 예측
2. **Bipartite Matching Loss**: 예측 N개와 GT M개를 Hungarian으로 1:1 매칭 (N>M)
3. **No NMS**: 중복 예측이 없음 — 각 query가 서로 다른 객체를 담당

## 구현 순서
1. 합성 Detection 데이터셋
2. DETR 아키텍처 (CNN backbone + Transformer Enc-Dec + Prediction Head)
3. Hungarian Matching Loss
4. 학습 + mAP 평가
5. DETR vs YOLO 비교 (anchor-free vs anchor-based)

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.optimize import linear_sum_assignment
import math, time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'디바이스: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

IMG_H, IMG_W = 256, 256
N_CLASSES    = 4   # car, pedestrian, cyclist, truck
N_QUERIES    = 20  # object queries 수 (예측 슬롯)
CLASS_NAMES  = ['car', 'pedestrian', 'cyclist', 'truck']

## 1. 합성 Detection 데이터셋 (COCO 포맷)

In [ ]:
class DetectionDataset(Dataset):
    """
    합성 객체 검출 데이터셋.
    bbox: [cx, cy, w, h] 정규화 형식 (DETR 표준)
    """
    SHAPES = {
        0: (0.12, 0.07),  # car: 가로 12%, 세로 7%
        1: (0.04, 0.10),  # pedestrian
        2: (0.05, 0.08),  # cyclist
        3: (0.16, 0.09),  # truck
    }
    COLORS = {
        0: (0.8, 0.2, 0.2),  # car: 붉은색
        1: (0.2, 0.8, 0.2),  # pedestrian: 초록
        2: (0.2, 0.2, 0.8),  # cyclist: 파랑
        3: (0.8, 0.8, 0.2),  # truck: 노랑
    }

    def __init__(self, n_samples=800, h=IMG_H, w=IMG_W, max_objs=8):
        self.n = n_samples
        self.h, self.w = h, w
        self.max_objs = max_objs

    def _make_scene(self, seed):
        rng = np.random.default_rng(seed)
        img = np.ones((self.h, self.w, 3), dtype=np.float32)
        # 배경 그라디언트
        bg = rng.uniform([0.5, 0.5, 0.5], [0.8, 0.8, 0.8])
        img[:] = bg
        # 노이즈 텍스처
        img += rng.uniform(-0.05, 0.05, img.shape)

        n_objs = rng.integers(1, self.max_objs + 1)
        boxes, labels = [], []
        for _ in range(n_objs):
            cls = rng.integers(0, N_CLASSES)
            rel_w, rel_h = self.SHAPES[cls]
            rel_w *= rng.uniform(0.7, 1.3)
            rel_h *= rng.uniform(0.7, 1.3)
            bw = int(rel_w * self.w)
            bh = int(rel_h * self.h)
            bw = max(8, min(bw, self.w // 2))
            bh = max(8, min(bh, self.h // 2))
            x1 = rng.integers(0, self.w - bw)
            y1 = rng.integers(0, self.h - bh)
            x2, y2 = x1 + bw, y1 + bh

            color = rng.uniform(np.array(self.COLORS[cls]) - 0.15,
                                np.array(self.COLORS[cls]) + 0.15)
            color = np.clip(color, 0, 1)
            img[y1:y2, x1:x2] = color
            # 간단한 내부 텍스처
            img[y1:y2:2, x1:x2] *= 0.9

            cx = (x1 + x2) / 2 / self.w
            cy = (y1 + y2) / 2 / self.h
            nw = bw / self.w
            nh = bh / self.h
            boxes.append([cx, cy, nw, nh])
            labels.append(cls)

        img = np.clip(img, 0, 1)
        return img, np.array(boxes, dtype=np.float32), np.array(labels, dtype=np.int64)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        img, boxes, labels = self._make_scene(idx)
        img_t    = torch.from_numpy(img.transpose(2,0,1)).float()
        boxes_t  = torch.from_numpy(boxes).float()
        labels_t = torch.from_numpy(labels).long()
        return img_t, boxes_t, labels_t


def detr_collate(batch):
    imgs   = torch.stack([b[0] for b in batch])
    boxes  = [b[1] for b in batch]
    labels = [b[2] for b in batch]
    return imgs, boxes, labels


# 데이터 확인
dataset = DetectionDataset(n_samples=1000)
img_s, boxes_s, labels_s = dataset[3]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, idx in zip(axes, [0, 3, 7]):
    img_v, boxes_v, labels_v = dataset[idx]
    ax.imshow(img_v.permute(1,2,0).numpy())
    for box, lbl in zip(boxes_v, labels_v):
        cx, cy, bw, bh = box.numpy()
        x1 = (cx - bw/2) * IMG_W
        y1 = (cy - bh/2) * IMG_H
        rect = patches.Rectangle((x1, y1), bw*IMG_W, bh*IMG_H,
                                  linewidth=2, edgecolor='white', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-3, CLASS_NAMES[lbl], color='white', fontsize=8,
                bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))
    ax.axis('off')
    ax.set_title(f'샘플 #{idx} ({len(boxes_v)}개 객체)')
plt.tight_layout()
plt.show()
print(f'데이터셋: {len(dataset)}개')

## 2. DETR 아키텍처

```
Image (H,W,3)
  → CNN Backbone        : (H/32, W/32, C) feature map
  → Positional Encoding : 2D 위치 정보 추가
  → Transformer Encoder : 전역 컨텍스트 집계
  → Transformer Decoder : N_QUERIES learnable query
  → Prediction Head     : N_QUERIES × (N_CLASSES+1 + 4)
                          (no_object 포함 클래스 + cxcywh bbox)
```

In [ ]:
# ── 2D Sinusoidal Positional Encoding ────────────────────────────────────
class PositionEncoding2D(nn.Module):
    """
    2D sin/cos positional encoding.
    feature map (B, C, H, W) → (B, C, H, W) + pos
    """
    def __init__(self, d_model, temperature=10000, normalize=True):
        super().__init__()
        self.d_model = d_model
        self.temperature = temperature
        self.normalize = normalize

    def forward(self, x):
        B, C, H, W = x.shape
        y_embed = torch.arange(H, device=x.device, dtype=torch.float32).unsqueeze(1).expand(H, W)
        x_embed = torch.arange(W, device=x.device, dtype=torch.float32).unsqueeze(0).expand(H, W)
        if self.normalize:
            y_embed = y_embed / (H + 1e-6) * 2 * math.pi
            x_embed = x_embed / (W + 1e-6) * 2 * math.pi

        d_half = C // 4
        dim_t  = torch.arange(d_half, device=x.device, dtype=torch.float32)
        dim_t  = self.temperature ** (2 * (dim_t // 2) / d_half)

        pos_x = x_embed[:, :, None] / dim_t
        pos_y = y_embed[:, :, None] / dim_t

        pos_x = torch.stack([pos_x[:,:,0::2].sin(), pos_x[:,:,1::2].cos()], dim=3).flatten(2)
        pos_y = torch.stack([pos_y[:,:,0::2].sin(), pos_y[:,:,1::2].cos()], dim=3).flatten(2)

        pos = torch.cat([pos_y, pos_x], dim=2).permute(2, 0, 1).unsqueeze(0)  # (1, C, H, W)
        # C 맞추기
        if pos.shape[1] != C:
            pos = F.pad(pos, (0,0,0,0,0,C-pos.shape[1]))
        return x + pos


# ── CNN Backbone ─────────────────────────────────────────────────────────
class DETRBackbone(nn.Module):
    """이미지 (B,3,H,W) → (B,d_model,H/32,W/32)"""
    def __init__(self, d_model=256):
        super().__init__()
        self.net = nn.Sequential(
            # stride 2×5 = 1/32
            nn.Conv2d(3,   32,  3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32),  nn.ReLU(True),
            nn.Conv2d(32,  64,  3, stride=2, padding=1, bias=False), nn.BatchNorm2d(64),  nn.ReLU(True),
            nn.Conv2d(64,  128, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, d_model, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(d_model), nn.ReLU(True),
        )
        self.pos_enc = PositionEncoding2D(d_model)

    def forward(self, x):
        feat = self.net(x)           # (B, d_model, H/32, W/32)
        feat = self.pos_enc(feat)    # + positional encoding
        B, C, H, W = feat.shape
        # Transformer 입력 형태: (H*W, B, C)
        feat_flat = feat.flatten(2).permute(2, 0, 1)
        return feat_flat, (H, W)


# ── DETR 전체 모델 ────────────────────────────────────────────────────────
class DETR(nn.Module):
    def __init__(self, n_classes=N_CLASSES, n_queries=N_QUERIES, d_model=128,
                 nhead=4, num_enc=2, num_dec=2):
        super().__init__()
        self.n_queries = n_queries
        self.backbone  = DETRBackbone(d_model)

        # Transformer (PyTorch 내장)
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=d_model*4,
                                               dropout=0.1, batch_first=False)
        dec_layer = nn.TransformerDecoderLayer(d_model, nhead, dim_feedforward=d_model*4,
                                               dropout=0.1, batch_first=False)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_enc)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers=num_dec)

        # Object queries: (N_QUERIES, 1, d_model) — 학습되는 슬롯
        self.query_embed = nn.Embedding(n_queries, d_model)

        # Prediction heads
        self.class_head = nn.Linear(d_model, n_classes + 1)  # +1: no-object
        self.bbox_head  = nn.Sequential(
            nn.Linear(d_model, d_model), nn.ReLU(),
            nn.Linear(d_model, d_model), nn.ReLU(),
            nn.Linear(d_model, 4)  # cx, cy, w, h ∈ [0,1]
        )

    def forward(self, x):
        B = x.shape[0]
        # Backbone + positional encoding
        src, hw = self.backbone(x)  # (HW, B, d_model)

        # Encoder
        memory = self.encoder(src)  # (HW, B, d_model)

        # Object queries (N_QUERIES, B, d_model)
        queries = self.query_embed.weight.unsqueeze(1).expand(-1, B, -1)

        # Decoder: queries attend to encoder memory
        hs = self.decoder(queries, memory)  # (N_QUERIES, B, d_model)
        hs = hs.permute(1, 0, 2)            # (B, N_QUERIES, d_model)

        # Predictions
        class_logit = self.class_head(hs)                  # (B, N_QUERIES, N_CLASSES+1)
        bbox_pred   = self.bbox_head(hs).sigmoid()          # (B, N_QUERIES, 4) ∈ [0,1]
        return class_logit, bbox_pred


model = DETR().to(device)
total = sum(p.numel() for p in model.parameters())
print(f'DETR 파라미터: {total/1e6:.2f}M')
x_dummy = torch.randn(2, 3, IMG_H, IMG_W).to(device)
cls_out, box_out = model(x_dummy)
print(f'클래스 logit: {cls_out.shape}')   # (2, 20, 5)
print(f'박스 예측:     {box_out.shape}')   # (2, 20, 4)

## 3. Hungarian Matching Loss

**핵심 원리:**
- 예측 N_QUERIES개 vs GT M개 (M ≤ N_QUERIES)
- 비용 행렬 = -(클래스 확률) + λ1·L1(bbox) + λ2·GIoU(bbox)
- Hungarian 알고리즘으로 최소 비용 1:1 매칭
- 매칭된 쌍만 loss 계산, 나머지 쿼리는 no-object 예측 유도

In [ ]:
def box_cxcywh_to_xyxy(boxes):
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx-w/2, cy-h/2, cx+w/2, cy+h/2], dim=-1)

def generalized_iou(boxes1, boxes2):
    """
    GIoU loss — DETR 논문에서 사용.
    boxes: [x1, y1, x2, y2] 형식
    """
    x1 = torch.max(boxes1[:,0], boxes2[:,0])
    y1 = torch.max(boxes1[:,1], boxes2[:,1])
    x2 = torch.min(boxes1[:,2], boxes2[:,2])
    y2 = torch.min(boxes1[:,3], boxes2[:,3])
    inter = (x2-x1).clamp(0) * (y2-y1).clamp(0)
    area1 = (boxes1[:,2]-boxes1[:,0]) * (boxes1[:,3]-boxes1[:,1])
    area2 = (boxes2[:,2]-boxes2[:,0]) * (boxes2[:,3]-boxes2[:,1])
    union = area1 + area2 - inter
    iou   = inter / (union + 1e-6)
    # 최소 외접 박스
    enc_x1 = torch.min(boxes1[:,0], boxes2[:,0])
    enc_y1 = torch.min(boxes1[:,1], boxes2[:,1])
    enc_x2 = torch.max(boxes1[:,2], boxes2[:,2])
    enc_y2 = torch.max(boxes1[:,3], boxes2[:,3])
    enc_area = (enc_x2-enc_x1).clamp(0) * (enc_y2-enc_y1).clamp(0)
    giou = iou - (enc_area - union) / (enc_area + 1e-6)
    return giou


class HungarianMatcher(nn.Module):
    """예측과 GT를 Hungarian 최적 매칭."""
    def __init__(self, cost_class=1.0, cost_bbox=5.0, cost_giou=2.0):
        super().__init__()
        self.cost_class = cost_class
        self.cost_bbox  = cost_bbox
        self.cost_giou  = cost_giou

    @torch.no_grad()
    def forward(self, class_logit, bbox_pred, gt_boxes_list, gt_labels_list):
        """
        반환: [(pred_idx, gt_idx), ...] 배치별 매칭 인덱스
        """
        B, N, _ = class_logit.shape
        indices = []

        for b in range(B):
            gt_boxes  = gt_boxes_list[b].to(class_logit.device)
            gt_labels = gt_labels_list[b].to(class_logit.device)
            M = len(gt_boxes)
            if M == 0:
                indices.append((torch.tensor([], dtype=torch.long),
                                torch.tensor([], dtype=torch.long)))
                continue

            # 클래스 비용: -softmax[gt_class] 확률
            prob = class_logit[b].softmax(-1)  # (N, C+1)
            cost_cls = -prob[:, gt_labels]     # (N, M)

            # BBox L1 비용
            cost_l1 = torch.cdist(bbox_pred[b], gt_boxes, p=1)  # (N, M)

            # GIoU 비용
            pred_xyxy = box_cxcywh_to_xyxy(bbox_pred[b])   # (N, 4)
            gt_xyxy   = box_cxcywh_to_xyxy(gt_boxes)       # (M, 4)
            # 모든 쌍에 대해 GIoU 계산
            cost_giou_mat = torch.zeros(N, M, device=class_logit.device)
            for n in range(N):
                pred_rep = pred_xyxy[n].unsqueeze(0).expand(M, -1)
                cost_giou_mat[n] = -generalized_iou(pred_rep, gt_xyxy)

            cost = (self.cost_class * cost_cls +
                    self.cost_bbox  * cost_l1  +
                    self.cost_giou  * cost_giou_mat)  # (N, M)

            ri, ci = linear_sum_assignment(cost.cpu().numpy())
            indices.append((torch.tensor(ri, dtype=torch.long),
                            torch.tensor(ci, dtype=torch.long)))
        return indices


class DETRLoss(nn.Module):
    def __init__(self, n_classes=N_CLASSES, no_obj_weight=0.1):
        super().__init__()
        self.matcher = HungarianMatcher()
        # no-object 클래스 가중치 낮춤 (배경 슬롯이 많으므로)
        weight = torch.ones(n_classes + 1)
        weight[-1] = no_obj_weight
        self.register_buffer('weight', weight)

    def forward(self, class_logit, bbox_pred, gt_boxes_list, gt_labels_list):
        B, N, _ = class_logit.shape
        indices = self.matcher(class_logit, bbox_pred, gt_boxes_list, gt_labels_list)

        # ── 클래스 Loss ─────────────────────────────────────────────────
        # 기본 no-object 타깃
        target_cls = torch.full((B, N), N_CLASSES, dtype=torch.long, device=class_logit.device)
        for b, (pred_idx, gt_idx) in enumerate(indices):
            if len(pred_idx) > 0:
                target_cls[b, pred_idx] = gt_labels_list[b][gt_idx].to(class_logit.device)
        loss_cls = F.cross_entropy(class_logit.reshape(-1, N_CLASSES+1),
                                   target_cls.reshape(-1), weight=self.weight)

        # ── BBox Loss (매칭된 쌍만) ───────────────────────────────────────
        loss_bbox, loss_giou = torch.tensor(0., device=class_logit.device), \
                               torch.tensor(0., device=class_logit.device)
        n_matched = 0
        for b, (pred_idx, gt_idx) in enumerate(indices):
            if len(pred_idx) == 0:
                continue
            pred_b = bbox_pred[b][pred_idx]
            gt_b   = gt_boxes_list[b][gt_idx].to(class_logit.device)
            loss_bbox += F.l1_loss(pred_b, gt_b, reduction='sum')
            pred_xyxy = box_cxcywh_to_xyxy(pred_b)
            gt_xyxy   = box_cxcywh_to_xyxy(gt_b)
            giou = generalized_iou(pred_xyxy, gt_xyxy)
            loss_giou += (1 - giou).sum()
            n_matched += len(pred_idx)

        if n_matched > 0:
            loss_bbox /= n_matched
            loss_giou /= n_matched

        loss = loss_cls + 5.0 * loss_bbox + 2.0 * loss_giou
        return loss, loss_cls.item(), loss_bbox.item(), loss_giou.item()


criterion = DETRLoss().to(device)
print('Hungarian Matcher + DETR Loss 구성 완료')

## 4. 학습

In [ ]:
n_val = 150
train_ds, val_ds = random_split(dataset, [len(dataset)-n_val, n_val],
                                generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,
                          collate_fn=detr_collate, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False,
                          collate_fn=detr_collate, num_workers=0)
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

# DETR는 backbone과 transformer를 다른 lr로 학습
param_dicts = [
    {'params': [p for n,p in model.named_parameters() if 'backbone' not in n], 'lr': 1e-4},
    {'params': [p for n,p in model.named_parameters() if 'backbone' in n],     'lr': 1e-5},
]
optimizer = torch.optim.AdamW(param_dicts, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

EPOCHS = 40
history = {'train_loss': [], 'val_cls_loss': [], 'val_bbox_loss': []}

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for imgs, boxes, labels in train_loader:
        imgs = imgs.to(device)
        cls_logit, bbox_pred = model(imgs)
        loss, lc, lb, lg = criterion(cls_logit, bbox_pred, boxes, labels)
        optimizer.zero_grad()
        loss.backward()
        # gradient clipping (Transformer 학습 안정성)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()
    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_cls, val_bbox = 0.0, 0.0
    with torch.no_grad():
        for imgs, boxes, labels in val_loader:
            cls_logit, bbox_pred = model(imgs.to(device))
            _, lc, lb, _ = criterion(cls_logit, bbox_pred, boxes, labels)
            val_cls  += lc
            val_bbox += lb

    val_cls  /= len(val_loader)
    val_bbox /= len(val_loader)
    history['train_loss'].append(train_loss)
    history['val_cls_loss'].append(val_cls)
    history['val_bbox_loss'].append(val_bbox)

    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1:2d}/{EPOCHS} | Loss: {train_loss:.3f} | '
              f'Val Cls: {val_cls:.3f} | Val BBox: {val_bbox:.4f}')

print('\n학습 완료!')

## 5. mAP 평가

In [ ]:
def compute_iou_single(box1, box2):
    """cxcywh → xyxy 변환 후 IoU."""
    b1 = [box1[0]-box1[2]/2, box1[1]-box1[3]/2, box1[0]+box1[2]/2, box1[1]+box1[3]/2]
    b2 = [box2[0]-box2[2]/2, box2[1]-box2[3]/2, box2[0]+box2[2]/2, box2[1]+box2[3]/2]
    ix1, iy1 = max(b1[0],b2[0]), max(b1[1],b2[1])
    ix2, iy2 = min(b1[2],b2[2]), min(b1[3],b2[3])
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    a1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
    a2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
    return inter / (a1 + a2 - inter + 1e-6)


def compute_map(model, loader, n_classes=N_CLASSES, conf_thresh=0.5, iou_thresh=0.5):
    model.eval()
    all_preds = {c: [] for c in range(n_classes)}  # (score, tp/fp)
    all_gt    = {c: 0  for c in range(n_classes)}  # GT count

    with torch.no_grad():
        for imgs, boxes_list, labels_list in loader:
            cls_logit, bbox_pred = model(imgs.to(device))
            prob = cls_logit.softmax(-1).cpu()
            bbox = bbox_pred.cpu()

            for b in range(imgs.shape[0]):
                gt_boxes  = boxes_list[b].numpy()
                gt_labels = labels_list[b].numpy()
                for c in gt_labels:
                    all_gt[c] += 1

                matched = np.zeros(len(gt_boxes), dtype=bool)
                # 신뢰도 높은 예측만
                scores, cls_ids = prob[b, :, :n_classes].max(-1)
                keep = scores > conf_thresh
                pred_boxes  = bbox[b][keep].numpy()
                pred_scores = scores[keep].numpy()
                pred_cls    = cls_ids[keep].numpy()

                order = np.argsort(-pred_scores)
                for i in order:
                    c_id = pred_cls[i]
                    gt_c_idx = np.where(gt_labels == c_id)[0]
                    best_iou, best_j = -1, -1
                    for j in gt_c_idx:
                        iou = compute_iou_single(pred_boxes[i], gt_boxes[j])
                        if iou > best_iou:
                            best_iou, best_j = iou, j
                    if best_iou >= iou_thresh and not matched[best_j]:
                        all_preds[c_id].append((pred_scores[i], 1))
                        matched[best_j] = True
                    else:
                        all_preds[c_id].append((pred_scores[i], 0))

    # AP per class
    aps = []
    for c in range(n_classes):
        if all_gt[c] == 0:
            continue
        preds = sorted(all_preds[c], key=lambda x: -x[0])
        tp_list, fp_list = [], []
        for _, is_tp in preds:
            tp_list.append(is_tp)
            fp_list.append(1 - is_tp)
        tp_cum = np.cumsum(tp_list)
        fp_cum = np.cumsum(fp_list)
        rec = tp_cum / (all_gt[c] + 1e-6)
        pre = tp_cum / (tp_cum + fp_cum + 1e-6)
        # 11-point interpolation
        ap = 0.0
        for t in np.linspace(0, 1, 11):
            p = pre[rec >= t].max() if (rec >= t).any() else 0
            ap += p / 11
        aps.append((c, ap))

    return aps, np.mean([a for _, a in aps]) if aps else 0.0


aps, map50 = compute_map(model, val_loader)
print('\n클래스별 AP@0.5:')
for cls_id, ap in aps:
    print(f'  {CLASS_NAMES[cls_id]:<12}: {ap:.4f}')
print(f'  mAP@0.5: {map50:.4f}')

## 6. 정성적 결과 + Object Query 시각화

In [ ]:
CLASS_COLORS = ['red', 'lime', 'deepskyblue', 'yellow']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
model.eval()
samples = [val_ds[i] for i in [0, 5, 10]]

with torch.no_grad():
    for col, (img_s, boxes_s, labels_s) in enumerate(samples):
        cls_logit, bbox_pred = model(img_s.unsqueeze(0).to(device))
        prob = cls_logit[0].softmax(-1).cpu()
        bbox = bbox_pred[0].cpu()

        # GT 표시
        axes[0, col].imshow(img_s.permute(1,2,0).numpy())
        axes[0, col].set_title(f'GT ({len(boxes_s)}개)', fontsize=9)
        for box, lbl in zip(boxes_s, labels_s):
            cx, cy, bw, bh = box.numpy()
            x1, y1 = (cx-bw/2)*IMG_W, (cy-bh/2)*IMG_H
            rect = patches.Rectangle((x1,y1), bw*IMG_W, bh*IMG_H,
                                      lw=2, ec='white', fc='none')
            axes[0, col].add_patch(rect)
            axes[0, col].text(x1, y1-3, CLASS_NAMES[lbl], color='white',
                              fontsize=7, bbox=dict(fc='black', alpha=0.6, pad=1))

        # Prediction 표시 (신뢰도 > 0.5)
        axes[1, col].imshow(img_s.permute(1,2,0).numpy())
        scores, cls_ids = prob[:, :N_CLASSES].max(-1)
        keep = scores > 0.5
        n_det = keep.sum().item()
        axes[1, col].set_title(f'DETR 예측 ({n_det}개)', fontsize=9)
        for i in range(N_QUERIES):
            if not keep[i]:
                continue
            cx, cy, bw, bh = bbox[i].numpy()
            x1, y1 = (cx-bw/2)*IMG_W, (cy-bh/2)*IMG_H
            c_id = cls_ids[i].item()
            rect = patches.Rectangle((x1,y1), bw*IMG_W, bh*IMG_H,
                                      lw=2, ec=CLASS_COLORS[c_id], fc='none')
            axes[1, col].add_patch(rect)
            axes[1, col].text(x1, y1-3,
                              f'{CLASS_NAMES[c_id]} {scores[i]:.2f}',
                              color='white', fontsize=7,
                              bbox=dict(fc=CLASS_COLORS[c_id], alpha=0.7, pad=1))
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 7. 학습 곡선

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'],   'b-'); axes[0].set_title('Train Loss'); axes[0].grid(True)
axes[1].plot(history['val_cls_loss'], 'r-'); axes[1].set_title('Val Class Loss (CE)'); axes[1].grid(True)
axes[2].plot(history['val_bbox_loss'],'g-'); axes[2].set_title('Val BBox Loss (L1)'); axes[2].grid(True)
plt.tight_layout()
plt.show()

## 8. 최종 결과 요약

In [ ]:
print('=' * 65)
print('  skillup_round7 / 03 DETR — 최종 결과')
print('=' * 65)
print(f'  모델:     DETR (CNN backbone + Transformer Enc-Dec + Object Query)')
print(f'  데이터:   합성 자율주행 장면 (train=850, val=150, 4클래스)')
print(f'  mAP@0.5: {map50:.4f}')
print()
print('  클래스별 AP@0.5:')
for cls_id, ap in aps:
    print(f'    {CLASS_NAMES[cls_id]:<12}: {ap:.4f}')
print()
print('  DETR vs YOLO 핵심 차이:')
print('    YOLO: anchor 사전정의 + NMS 후처리 + threshold 튜닝 필요')
print('    DETR: anchor-free + NMS 불필요 + Hungarian 최적 매칭')
print('    DETR 단점: 학습 수렴 느림(500ep), 소형 객체 약함')
print('    → Deformable DETR로 해결: 다중 스케일 + deformable attention')
print()
print('  구현 핵심:')
print('    1. Object Query — N개 learnable 슬롯, 각각 하나의 객체 담당')
print('    2. Hungarian Matching — 예측N vs GT M 최적 1:1 매칭')
print('    3. GIoU Loss — L1 단독 대비 박스 형태 정확도 개선')
print('    4. 2D Positional Encoding — spatial 위치 정보 주입')
print('    5. no-object 가중치 — 배경 슬롯 과도한 loss 방지')
print('=' * 65)